<a href="https://colab.research.google.com/github/RUBADAYYEH/PalTechDome-Quantum-Safe-Encryption-for-Palestine-s-future/blob/main/PalTechDome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This is a hybrid RSA + AES encryption prototype (not quantum resilliant).


---


*RSA will encrypt/decrypt the AES key.*

*AES will encrypt the actual message efficiently.*



***FLOW:***



1.   Generate RSA key pair (for key exchange).
2.   Generate random AES key + IV (for actual message encryption).
3.   Encrypt message with AES-CBC.
4.   Encrypt AES key with RSA (envelope encryption).
5.   Send/store both encrypted message + encrypted AES key.
6.   Recipient uses RSA private key to recover AES key.
7.   Decrypt message with AES + IV.
8.   Remove padding → get original message.


In [ ]:
!pip install cryptography

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import hashes
import os

# RSA private key
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key() # The public key is generated from the private key (for more information review RSA algorithm).

#Generating AES key & IV for AES CBC
aes_key = os.urandom(32)  # 256 bit AES key
iv = os.urandom(16)       # 128 bit IV for AES-CBC (AES is block level encryption, even if we encrypt the same text twice it will give different cipher in this case)

#Encrypting message with AES
message = b"Hello From Team 1 ! This is a secret message."

cipher = Cipher(algorithms.AES(aes_key), modes.CBC(iv))
encryptor = cipher.encryptor()

# AES requires message length to be multiple of block size (16 bytes)
from cryptography.hazmat.primitives import padding as sym_padding

padder = sym_padding.PKCS7(128).padder()
padded_data = padder.update(message) + padder.finalize()

encrypted_message = encryptor.update(padded_data) + encryptor.finalize()

#Encrypting AES key with RSA (in envelope encryption method)

encrypted_aes_key = public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# NOTE: Reciver has the encrypted AES Key and the cipher text + iv.

#Reciver Decrypting AES key with RSA private key.
decrypted_aes_key = private_key.decrypt(
    encrypted_aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

#Decrypting message with AES key
cipher = Cipher(algorithms.AES(decrypted_aes_key), modes.CBC(iv))
decryptor = cipher.decryptor()
decrypted_padded = decryptor.update(encrypted_message) + decryptor.finalize()

# Unpad the message
unpadder = sym_padding.PKCS7(128).unpadder()
decrypted_message = unpadder.update(decrypted_padded) + unpadder.finalize()

print("Decrypted message:", decrypted_message.decode())



196885919096479284658697735966067222010826628942878994562381289782233202330247606773367392709903670698701930785657822595708717112009975819460073174364436697322966741572774179417521785681064936571946747440169339343006421781893613306970475491002755797965792304152571021227908215430368202918247880342943739776890777454434888608207645826960254625448158268275136103572851155022354965351977690435592915481844907686284863030447742321326802345182503814262769185753871522133501837999747242594409653087520904398504894511544285047975794197118784617528514929523379869784981113518386378456874356786744033847910931612630204032513
Decrypted message: Hello From Team 1 ! This is a secret message.


# Hybrid Key Establishment: RSA + Kyber → HKDF → Encryption (AES)

Derive the AES session key from two independent secrets, one from RSA and one from Kyber, using a KDF.

**An attacker must break both RSA and Kyber to recover the AES key.**

Alice = receiver (server)

Bob = sender (client)

Alice has: RSA keypair (pk_RSA, sk_RSA), Kyber keypair (pk_KYB, sk_KYB).

***ENCRYPTION FLOW:***



1.   Bob creates an RSA secret (classical).
2.   Bob then encrypts it with Alice’s RSA public key.
3.   Bob creates a Kyber secret (post-quantum).
4.   Bob combines secrets using HKDF (This produces a uniform, strong AES key).
5.   Bob encrypts data with AES.
6.   Bob sends Alice:RSA secret, Kyber cipher secret, AES_nonce, AES_ciphertext.

***DECRYPTION FLOW:***



1.   Alice recovers RSA secret using her private RSA key.
2.   Alice recovers Kyber secret using her private Kyber key.
3.   Alice re-derives AES key using HKDF.
4.   Alice decrypts data.



In [ ]:
!pip install pqcrypto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 33.4 MB/s eta 0:00:00


In [ ]:
import os

from cryptography.hazmat.primitives.asymmetric import rsa, padding as rsa_padding
from cryptography.hazmat.primitives import hashes
from pqcrypto.kem.ml_kem_1024 import generate_keypair, encrypt, decrypt
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# RSA keypair
rsa_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=3072,
)
rsa_public_key = rsa_private_key.public_key()

# Kyber keypair
kyber_public_key, kyber_private_key = generate_keypair()

#Message encryption

message = b"Hello from Team 1! this is our secret message"

#Generating RSA Shared secret
rsa_secret = os.urandom(32)

rsa_ciphertext = rsa_public_key.encrypt(
    rsa_secret,
    rsa_padding.OAEP(
        mgf=rsa_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

#Kyber encapsulation
kyber_ciphertext, kyber_shared_secret = encrypt(kyber_public_key)

#Combine secrets via HKDF
combined_secret = rsa_secret + kyber_shared_secret

hkdf = HKDF(
    algorithm=hashes.SHA256(),
    length=32,  # AES-256
    salt=None,
    info=b"rsa-kyber-hybrid-session-key",
)

aes_key = hkdf.derive(combined_secret)

#AES-GCM encryption
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)

ciphertext = aesgcm.encrypt(nonce, message, None)


#DECRYPTION

#Recover RSA secret
rsa_secret_rec = rsa_private_key.decrypt(
    rsa_ciphertext,
    rsa_padding.OAEP(
        mgf=rsa_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

#Recover Kyber secret
kyber_shared_secret_rec = decrypt(kyber_private_key, kyber_ciphertext)

#Re-derive AES key
combined_secret_rec = rsa_secret_rec + kyber_shared_secret_rec

hkdf_rec = HKDF(
    algorithm=hashes.SHA256(),
    length=32,
    salt=None,
    info=b"rsa-kyber-hybrid-session-key",
)

aes_key_rec = hkdf_rec.derive(combined_secret_rec)

#AES-GCM decryption --------
aesgcm_rec = AESGCM(aes_key_rec)
decrypted_message = aesgcm_rec.decrypt(nonce, ciphertext, None)

print("Decrypted message:", decrypted_message.decode())


Decrypted message: Hello from Team 1! this is our secret message
